In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l5.6 Perplexity and evaluation

Loss is in nats; **perplexity** = `exp(loss)` is the readable version: the
effective number of equally-likely next tokens. A model guessing uniformly over
V tokens has perplexity exactly V, so beating V is the bar.

In [ ]:
from pocketlm import (CharTokenizer, PocketLM, PocketLMConfig, init_weights,
                      make_batch, estimate_loss, perplexity)

corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
data = torch.tensor(tok.encode(corpus), dtype=torch.long)
torch.manual_seed(0)
cfg = PocketLMConfig(vocab_size=tok.vocab_size, block_size=32, n_layer=2,
                     n_head=2, n_embd=64)
model = init_weights(PocketLM(cfg))
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
g = torch.Generator().manual_seed(0)
steps = 150 if os.environ.get("POCKETLM_CI") else 400
for _ in range(steps):
    x, y = make_batch(data, 32, 16, generator=g)
    _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
loss = estimate_loss(model, data, 32, 16, iters=20, generator=g)
print("loss:", round(loss, 3), " perplexity:", round(perplexity(loss), 2))
print("uniform baseline perplexity:", tok.vocab_size)

Even a micro model beats the uniform baseline, proof it learned real structure.
Always report perplexity against that baseline, not as a bare number.

In [ ]:
assert perplexity(loss) < tok.vocab_size